**INIT LIBRARIES**

In [1]:
from google.cloud import bigquery
import os
from dotenv import load_dotenv
from sklearn.metrics import mean_absolute_error
from sklearn.linear_model import LinearRegression

load_dotenv()

True

**ENVS**|

In [2]:
PROJECT = os.getenv("PROJECT").strip()
LOCATION = os.getenv("LOCATION").strip()
QUERY = """
    SELECT
        *
    FROM `trial-f1.openf1_marts.mart_ml_position_forecast`    
"""

**CONNECT CLIENT AND FETCH DATASET**

In [3]:
client = bigquery.Client(project=PROJECT, location=LOCATION)
df = client.query(QUERY).to_dataframe()

c:\Users\toras\anaconda3\envs\de\Lib\site-packages\google\cloud\bigquery\table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


**EDA**

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 841 entries, 0 to 840
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   meeting_key      841 non-null    Int64  
 1   driver_number    841 non-null    Int64  
 2   year             841 non-null    Int64  
 3   session_name     841 non-null    str    
 4   grid_position    833 non-null    Int64  
 5   best_lap_time    827 non-null    float64
 6   finish_position  841 non-null    Int64  
dtypes: Int64(5), float64(1), str(1)
memory usage: 53.9 KB


In [5]:
df = df.dropna()

In [6]:
df.info()

<class 'pandas.DataFrame'>
Index: 827 entries, 8 to 840
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   meeting_key      827 non-null    Int64  
 1   driver_number    827 non-null    Int64  
 2   year             827 non-null    Int64  
 3   session_name     827 non-null    str    
 4   grid_position    827 non-null    Int64  
 5   best_lap_time    827 non-null    float64
 6   finish_position  827 non-null    Int64  
dtypes: Int64(5), float64(1), str(1)
memory usage: 59.4 KB


In [7]:
df = df.sort_values(
    by=["meeting_key", "grid_position"],
    ascending=[True, True]
)

In [8]:
df.head()

,meeting_key,driver_number,year,session_name,grid_position,best_lap_time,finish_position
8,1254,4,2025,Race,1,75.096,1
49,1254,81,2025,Race,2,75.180,9
90,1254,1,2025,Race,3,75.481,2
131,1254,63,2025,Race,4,75.546,3
172,1254,22,2025,Race,5,75.670,12


In [9]:
race = df.loc[df["session_name"] == "Race"]
race

,meeting_key,driver_number,year,session_name,grid_position,best_lap_time,finish_position
8,1254,4,2025,Race,1,75.096,1
49,1254,81,2025,Race,2,75.180,9
90,1254,1,2025,Race,3,75.481,2
131,1254,63,2025,Race,4,75.546,3
172,1254,22,2025,Race,5,75.670,12
...,...,...,...,...,...,...,...
747,1288,23,2026,Race,18,68.509,17
787,1288,11,2026,Race,19,68.945,21
825,1288,77,2026,Race,20,69.030,22
834,1288,14,2026,Race,21,69.942,18


**TRAIN TEST SPLIT**

In [10]:
train = race[race["year"] == 2025]
test = race.loc[race["year"] == 2026]

**DEFINE FEATURES AND TARGETS**

In [11]:
FEATURES = ["grid_position", "best_lap_time"]
TARGET = "finish_position" \

X_train = train[FEATURES]
y_train = train[TARGET]

X_test = test[FEATURES]
y_test = test[TARGET]

In [12]:
baseline_pred = test["grid_position"]
baseline_mae = mean_absolute_error(y_test, baseline_pred)

print(f"Baseline (grid = finish) MAE: {baseline_mae:.2f}")

Baseline (grid = finish) MAE: 3.68


In [14]:
lr = LinearRegression()
lr.fit(X_train, y_train)

lr_pred = lr.predict(X_test)
lr_mae = mean_absolute_error(y_test, lr_pred)

print(f"LR MAE: {lr_mae:.2f}")
print(f"Coefficients: {dict(zip(FEATURES, lr.coef_))}")
print(f"Intercept: {lr.intercept_}")

LR MAE: 3.72
Coefficients: {'grid_position': np.float64(0.6692110246503244), 'best_lap_time': np.float64(-0.0015089618321812105)}
Intercept: 3.646198815045982
